# 05 · Deployment & Optimization
### *Aligning & Deploying LLMs — Unit 2*

A capable model is only useful if it **fits** and **runs**. This notebook covers the optimization toolkit from the deck:

1. **Quantization math** — why fewer bits shrink models dramatically.
2. **Measuring** a real model's size.
3. **Dynamic INT8 quantization** (CPU) — size + speed.
4. **Pruning** — remove redundant weights.
5. **Knowledge distillation** — a small student mimics a big teacher.
6. **4-bit** with bitsandbytes (GPU, optional) and **Ollama** for local serving.

> Parts 1–5 run on CPU. Part 6a needs a GPU.

In [ ]:
!pip -q install "transformers>=4.40" torch matplotlib

## 1 · Quantization memory math

Memory for the weights ≈ **params × bytes-per-param**. Dropping from FP32 (4 B) to INT4 (0.5 B) is an **8× cut**.

In [ ]:
import matplotlib.pyplot as plt

def weight_memory_gb(params_billion, bytes_per_param):
    return params_billion * bytes_per_param   # (B params · bytes) → GB

precisions = {"FP32": 4, "FP16": 2, "INT8": 1, "INT4": 0.5}
params_b = 7
mem = {p: weight_memory_gb(params_b, b) for p, b in precisions.items()}
for p, gb in mem.items():
    print(f"{p}: {gb:5.1f} GB")

plt.figure(figsize=(7, 4))
plt.bar(mem.keys(), mem.values(), color=["#6d5df0", "#8b7cf6", "#22cfe6", "#17a9c4"])
plt.ylabel("GB (weights only)"); plt.title(f"{params_b}B model memory by precision")
for i, v in enumerate(mem.values()):
    plt.text(i, v + 0.4, f"{v:.1f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

## 2 · Measure a real model

Let's count parameters and compute the FP32 vs FP16 footprint of `distilbert-base-uncased`. We use a
BERT-family model here because its layers are `nn.Linear` — the layer type PyTorch dynamic quantization and
bitsandbytes actually convert. (`distilgpt2` uses a custom `Conv1D`, which those tools skip.)

In [ ]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("distilbert-base-uncased")
n = sum(p.numel() for p in model.parameters())
print(f"parameters: {n/1e6:.1f} M")
print(f"FP32 weights: {n*4/1e6:.1f} MB")
print(f"FP16 weights: {n*2/1e6:.1f} MB")

## 3 · Dynamic INT8 quantization (CPU)

`torch.quantization.quantize_dynamic` casts the `Linear` layers to INT8 at inference time — a one-liner that shrinks
the model and often speeds up CPU inference, with a small accuracy cost.

In [ ]:
import os, time, copy
import torch.nn as nn

def disk_size_mb(m, path):
    torch.save(m.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return mb

fp32 = model.to("cpu").eval()
int8 = torch.quantization.quantize_dynamic(copy.deepcopy(fp32), {nn.Linear}, dtype=torch.qint8)

print(f"FP32 on disk: {disk_size_mb(fp32, 'fp32.pt'):.1f} MB")
print(f"INT8 on disk: {disk_size_mb(int8, 'int8.pt'):.1f} MB")

# rough latency check
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
ids = tok("The future of on-device AI is bright.", return_tensors="pt")

def timed(m, runs=20):
    with torch.no_grad():
        m(**ids)  # warmup
        t = time.time()
        for _ in range(runs): m(**ids)
    return (time.time() - t) / runs * 1000

print(f"FP32 latency: {timed(fp32):.1f} ms")
print(f"INT8 latency: {timed(int8):.1f} ms")

## 4 · Pruning

Pruning zeroes out the least-important weights, giving a **sparser** network of similar accuracy. Here we prune 50%
of one attention projection by magnitude and report the resulting sparsity.

In [ ]:
import torch.nn.utils.prune as prune

layer = model.transformer.layer[0].attention.q_lin   # an nn.Linear in the first block
prune.l1_unstructured(layer, name="weight", amount=0.5)

w = layer.weight
sparsity = (w == 0).float().mean().item()
print(f"weights pruned to zero: {sparsity*100:.1f}%")
prune.remove(layer, "weight")   # make the pruning permanent

## 5 · Knowledge distillation (concept)

A small **student** is trained to match a large **teacher's** soft output distribution (softened by a temperature `T`).
DistilBERT was built this way. Here is the core loss on toy logits.

In [ ]:
import torch, torch.nn.functional as F

T = 2.0                                   # temperature
teacher_logits = torch.tensor([[2.0, 0.5, -1.0, 0.2]])   # a confident teacher
student_logits = torch.tensor([[0.3, 0.1,  0.0, 0.1]], requires_grad=True)

# KL between softened distributions (the distillation loss)
loss = F.kl_div(
    F.log_softmax(student_logits / T, dim=-1),
    F.softmax(teacher_logits / T, dim=-1),
    reduction="batchmean",
) * (T * T)
print("distillation KL loss:", loss.item())

# One gradient step nudges the student toward the teacher's distribution
loss.backward()
print("student grad (direction to imitate teacher):", student_logits.grad)

## 6a · 4-bit loading with bitsandbytes (GPU only)

On a GPU, `bitsandbytes` loads weights in **4-bit** so large models fit in far less VRAM — the same trick Ollama uses.

In [ ]:
import torch
if torch.cuda.is_available():
    !pip -q install bitsandbytes
    from transformers import AutoModel, BitsAndBytesConfig
    cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    m4 = AutoModel.from_pretrained("distilbert-base-uncased", quantization_config=cfg, device_map="auto")
    print("Loaded in 4-bit. Footprint:",
          sum(p.numel() for p in m4.parameters()) , "params (stored 4-bit)")
else:
    print("No GPU detected — switch to a GPU runtime to try 4-bit loading.")

## 6b · Local deployment with Ollama

Colab can't run the Ollama background service, but on **your own machine** these commands give you a private, offline,
free LLM in one step:

```bash
# install (macOS/Linux):  curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3          # download quantized weights
ollama run mistral          # chat in the terminal
ollama serve                # start the local REST API on :11434

# call it from any language:
curl http://localhost:11434/api/generate -d '{"model":"mistral","prompt":"Explain quantization in one line."}'
```

Ollama ships models **pre-quantized** (usually 4-bit) — which is exactly why a 7B model that needed 28 GB at FP32
now runs in ~4 GB on a laptop.

## Recap & your turn

- **Quantization** is the workhorse — 8× smaller at INT4 for a small accuracy cost.
- **Pruning** removes redundancy; **distillation** trains a compact student.
- Choose deployment by constraints: **API** (fastest to ship), **cloud** (scale), **on-prem** (control), **local/Ollama** (private & free).

**Exercises**
1. Quantize a larger model (e.g. `gpt2-medium`) and compare size + latency.
2. Prune 90% of a layer — where does accuracy break?
3. Load a real quantized GGUF locally with `llama-cpp-python` and benchmark tokens/sec.